# Experimentations
This notebook permits to run experimentations to MLFlow locally

In [1]:
import mlflow
import mlflow.lightgbm
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import os
from mlflow.tracking import MlflowClient
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

# check tracking URI
assert "MLFLOW_TRACKING_URI" in os.environ, "MLFLOW_TRACKING_URI non définie"

experiment_name = "1-lightgbm"
seuil_utilisateur = 0.5 # pour optimiser F1-score 0.656, pour optimiser recall, mettre 0.3
random_state=42

client = MlflowClient()

exp = client.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(experiment_name)
else:
    exp_id = exp.experiment_id

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, make_scorer
from lightgbm import LGBMClassifier
import mlflow
import mlflow.lightgbm
import matplotlib.pyplot as plt

# Load prepared data
df = pd.read_csv("../.data/modelisation/app_train.csv")
target_col = "TARGET"
X = df.drop(columns=[target_col])
y = df[target_col]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

# Prepare scoring
scoring = {
    "auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

# run MLflow
with mlflow.start_run(experiment_id=exp_id):
    mlflow.log_artifact("2-experimentation-lightgbm.ipynb", artifact_path="notebooks")

    # Instantiate model with cost function (10x coeff)
    model = LGBMClassifier(n_estimators=100, random_state=random_state, class_weight={0: 1, 1: 10})

    # Cross-validation on training set
    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv_splitter,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    # Log params
    mlflow.log_params(model.get_params())

    # Fit final model on the whole training set (to evaluate on X_test)
    model.fit(X_train, y_train)
    
    # Predict probabilities on test set
    y_proba = model.predict_proba(X_test)[:, 1]

    # Appliquer le seuil
    y_pred_utilisateur = (y_proba >= seuil_utilisateur).astype(int)
    precision_util = precision_score(y_test, y_pred_utilisateur)
    recall_util = recall_score(y_test, y_pred_utilisateur)
    f1_util = f1_score(y_test, y_pred_utilisateur)
    roc_auc = roc_auc_score(y_test, y_pred_utilisateur)
    
    # Log les métriques avec ce seuil
    mlflow.log_metric("probability_threshold", round(seuil_utilisateur, 3))
    mlflow.log_metric("precision", round(precision_util, 3))
    mlflow.log_metric("recall", round(recall_util, 3))
    mlflow.log_metric("f1", round(f1_util, 3))
    mlflow.log_metric("auc", round(roc_auc, 3))

    # Log model
    try:
        mlflow.lightgbm.log_model(model.booster_, name="model")
    except TypeError:
        # fallback if signature changed
        mlflow.lightgbm.log_model(model.booster_, name="model")

[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025681 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12543
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 261
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.467789 -> initscore=-0.129021
[LightGBM] [Info] Start training from score -0.129021


2025/10/16 09:00:14 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run burly-asp-126 at: http://mlflow:5000/#/experiments/696573625829308745/runs/4497871705d845d59b60614f89e172db
🧪 View experiment at: http://mlflow:5000/#/experiments/696573625829308745


In [3]:

remote: Resolving deltas: 100% (6/6), completed with 4 local obje

SyntaxError: invalid syntax (2334553287.py, line 1)

[LightGBM] [Info] Number of positive: 15900, number of negative: 180906
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.539733 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12451
[LightGBM] [Info] Number of data points in the train set: 196806, number of used features: 260
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.467776 -> initscore=-0.129073
[LightGBM] [Info] Start training from score -0.129073
[LightGBM] [Info] Number of positive: 15901, number of negative: 180906
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.800667 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12541
[LightGBM] [Info] Number of data points in the train set: 196807, number of used features: 260
[LightGBM]